# Softmax as a custom autograd Function

Write softmax yourself, forward **and** backward, as a `torch.autograd.Function`, the way you would before fusing it into a kernel.

## Forward
Along dimension `dim`:
$$y_i = \frac{e^{x_i - m}}{\sum_j e^{x_j - m}}, \qquad m = \max_j x_j.$$
Subtracting the max keeps it finite for large logits. Don't call `torch.softmax`, `F.softmax` or `log_softmax`.

## Backward
Given the upstream gradient $g = \partial\mathcal L/\partial y$, the input gradient needs only the **output** $y$:
$$\frac{\partial\mathcal L}{\partial x_i} = y_i\Big(g_i - \sum_j g_j\, y_j\Big)$$
with the sum taken along the same `dim`. Save `y` (not `x`) for backward.

## What's graded
Write a class `Softmax(torch.autograd.Function)` with static `forward(ctx, x, dim)` and `backward(ctx, grad_y)`, and the function `softmax_and_grad(x, dim, g)`:

- `x`: float64 tensor of any shape with at least one dim; `dim`: int (may be negative); `g`: float64 tensor shaped like `x`.
- Return `(y, grad_x)`: `y = Softmax.apply(x, dim)` and the gradient of `(y * g).sum()` with respect to `x`.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

class Softmax(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, dim):
        e = x.exp()
        y = e / e.sum(dim=dim, keepdim=True)
        ctx.dim = dim
        ctx.save_for_backward(y)
        return y

    @staticmethod
    def backward(ctx, grad_y):
        (y,) = ctx.saved_tensors
        return grad_y * y, None          # TODO: this is only the diagonal of the Jacobian

def softmax_and_grad(x, dim, g):
    """Return (y, dL/dx) for L = (Softmax.apply(x, dim) * g).sum()."""
    x = x.detach().clone().requires_grad_()
    y = Softmax.apply(x, dim)
    (y * g).sum().backward()
    return y.detach(), x.grad


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [0.0, 0.0], "dtype": "float64"}, 0, {"__nd__": [1.0, 0.0], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [0.5, 0.5], "dtype": "float64"}, {"__nd__": [0.25, -0.25], "dtype": "float64"}]}}, {"a": [{"__nd__": [[1.0, 2.0, 3.0]], "dtype": "float64"}, -1, {"__nd__": [[0.0, 0.0, 1.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[0.09003057317038046, 0.24472847105479764, 0.6652409557748218]], "dtype": "float64"}, {"__nd__": [[-0.05989202454481893, -0.1628034019898044, 0.2226954265346234]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[1.8267565599574231, -3.0783319101980338, 0.9580639753088469, 0.06963722766094482], [1.3182500241810684, 0.385629249998389, 1.8272586275861753, 0.0317437591517664], [-0.5162294444924808, 0.5804849213397179, 0.43210686133773885, -0.35683935740335093]], "dtype": "float64"}, 0, {"__nd__": [[-0.24730382198818454, 0.7194406781853278, 0.7043159938619936, -0.4939342302351804], [-0.3677137240199963, -1.8067903895865323, 1.6792074674884705, -0.2242908783928769], [1.337277430495411, 0.4174655938071864, 1.9439627746249157, 1.537109976128164]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[0.589124699148744, 0.013935602354116264, 0.25151039642222506, 0.38231843656624326], [0.35429550995664805, 0.4451485497760148, 0.5998496128957633, 0.3681021188023905], [0.056579790894608094, 0.5409158478698689, 0.14863999068201172, 0.24957944463136617]], "dtype": "float64"}, {"__nd__": [[-0.02768567268021669, 0.01794753970851864, -0.19342390915331817, -0.23174735705654723], [-0.05931065926293907, -0.5512451940339379, 0.12347431276925384, -0.12387365962152769], [0.08699633194315581, 0.5332976543254193, 0.06994959638406419, 0.35562101667807494]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.318298439352904, 1.480763419858435, -0.9501235216099734, 1.2586181429890126, -1.4804236275921896], [0.3432363675742628, 1.064876469210243, 0.22363214164877185, -0.3671374972389881, -0.8055600446794365], [-0.3428000151424468, 1.051125142503177, 0.8908394462458891, -0.2621314629648405, -1.2460043473128628]], [[0.6739972418852872, -1.4498739364757562, -0.530854904621622, -0.7348283771922636, 0.7432652102376149], [0.23594974138056524, 0.46185473227162593, 0.27240791046479607, -0.6779301162200867, 0.535475983047878], [1.4124603684044394, -0.03677124294459777, 0.6335944644542391, -0.125903192963739, 1.0285554273542712]]], "dtype": "float64"}, 1, {"__nd__": [[[0.6666366521512817, 0.8758194707912133, 0.3484229955590589, 1.6400038821107275, -0.3611505786282483], [-0.33416607721805125, -0.5918599835407998, 0.610962701968474, -0.6224002707066544, -0.6445326365001358], [0.7274748296203505, 1.1620957720506857, 0.5743514885621694, -2.683103555466855, -0.9536324496718003]], [[-1.07348948063148, -1.2075111797268347, -0.4077676482371872, 1.8175432337026303, -0.3073985537659124], [-0.7354457743627153, 0.6988227448389438, 0.06917172943950622, 0.12068112866070582, -0.6866741120790538], [-0.0015044670379158913, 0.43453717981416906, -1.9113065771604514, -1.610224331010523, 1.0845605588518257]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[0.3934629455742241, 0.4328068843561231, 0.09490618881912041, 0.7065585189651189, 0.23652183341454613], [0.40339846702583876, 0.2855464415550866, 0.306937634150033, 0.13902452937118714, 0.4644733139443904], [0.20313858739993715, 0.28164667408879024, 0.5981561770308466, 0.15441695166369393, 0.29900485264106347]], [[0.26752188554701545, 0.084221474170463, 0.15535253797001702, 0.2566074858431682, 0.31821532121866214], [0.17263056998812637, 0.5697391332660119, 0.3468733708711128, 0.2716313689615939, 0.25851120447459935], [0.5598475444648582, 0.3460393925635251, 0.49777409115887006, 0.4717611451952378, 0.42327347430673856]]], "dtype": "float64"}, {"__nd__": [[[0.15398711631430476, 0.14648863063421422, -0.020473527493043462, 0.6939045821481186, 0.07303282337087241], [-0.24584676801825578, -0.322444060832153, 0.014369551272451549, -0.1779949689947029, 0.01179590371773792], [0.09185965170395095, 0.17595543019793886, 0.006103976220591986, -0.5159096131534157, -0.08482872708861029]], [[-0.176164460520367, -0.13932981278187942, 0.09056822330860798, 0.5332325440138217, -0.15628585069846807], [-0.05532139080898564, 0.14357851740478764, 0.36765959525176406, 0.10353132473085976, -0.22501020665891616], [0.2314858513293526, -0.004248704622908265, -0.4582278185603721, -0.6367638687446815, 0.3812960573573842]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[1000.0, 1001.0], [-1000.0, -1000.0]], "dtype": "float64"}, 1, {"__nd__": [[1.0, -1.0], [2.0, 5.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[0.2689414213699951, 0.7310585786300049], [0.5, 0.5]], "dtype": "float64"}, {"__nd__": [[0.3932238664829637, -0.3932238664829637], [-0.75, 0.75]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[3.0]], "dtype": "float64"}, -1, {"__nd__": [[7.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[1.0]], "dtype": "float64"}, {"__nd__": [[0.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[4.215022874620292, -2.5332835877956144, -5.101019522086095, -0.9277331285484722, -5.968261452206006, -1.7004849530311137], [8.255502119878635, 0.6603592175326625, 5.831442181773974, -1.1131026435467157, 1.847077809554949, 6.929152464996491], [-6.679687786111246, -1.7373899732491258, -0.7844925027271741, 1.4099058968268228, 7.76833361402197, 3.42763501789934], [-7.488332992590358, -1.3548602058967754, -4.468217077415087, -4.59523514981448, 10.124138575017565, -5.314853418737973]], "dtype": "float64"}, -1, {"__nd__": [[1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[0.9902563840075684, 0.0011614362908026175, 8.909264341871183e-05, 0.005784650131069017, 3.7428589424075616e-05, 0.002671008337717169], [0.737331722502762, 0.0003707953399058916, 0.06529908025035676, 6.29404099434961e-05, 0.0012148414554995818, 0.19572062004153232], [5.233909838340213e-07, 7.33227770261748e-05, 0.0001901415449452099, 0.0017064443679976377, 0.98519501462674, 0.012834553292307115], [2.2438603358911225e-08, 1.0344969340815289e-05, 4.598492668325012e-07, 4.049974142773361e-07, 0.9999885705367033, 1.9720867136920795e-07]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [-1.637205310218186e-16, -8.23331047574464e-20, -1.449930847615838e-17, -1.3975578459723105e-20, -2.6974899103295464e-19, -4.345870775280421e-17], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[0.23639517490996417, -0.7152122320787201, 0.8609809399519063, 0.35370323854134944], [0.1274277181041873, -0.07616714005920917, 1.1377169165755536, -0.9316998851391717], [-0.7269383253215163, -1.1166060472205357, 1.334546412220603, -0.2935663794448887]], [[-1.03740077070428, -0.4190734003279602, 2.499647141912049, -0.10747295455808797], [0.8500205977374702, 0.4651818264795043, -0.012383282080929063, -0.6927049513484409], [-0.33429524042418257, -0.3544524150979252, -2.209276174952708, 2.590132165299634]]], [[[0.9248362579408633, 0.11628650676709631, -0.20334913839455843, -0.7896443817288501], [0.949621715476415, -0.18685598546529084, 0.47927255908121963, -1.3874951680999685], [0.3784765367574817, 2.000301160167004, -0.28962886468938337, 0.7999549486302511]], [[0.03533363711049965, 1.1051905403205458, 1.0674245093302333, 0.06164637112836233], [-0.559585430021553, 0.2867541614425335, 0.04176902945224784, -0.6179033056301174], [0.45647884748874545, 0.4323228618761901, -0.8890266047821327, -0.1114878979154479]]]], "dtype": "float64"}, -2, {"__nd__": [[[[2.921139507657893, 0.8168344872544508, 0.09777931084435634, 2.511082373702184], [-0.45540319859640854, -2.47685411921746, -1.9069414995045355, -1.5195110137903052], [0.2600550542500467, 0.976356295342398, -2.170190238668888, 1.1172437209364225]], [[-1.2277384405743208, 0.25643484822861606, 0.463919602174066, 0.43232983369059774], [0.0075470802723938074, 0.17099551394253604, -1.139820149948546, 0.27821689039644487], [0.979652364759179, 0.9262226307647625, 2.6128529070721185, 1.0832325419401145]]], [[[0.7264273559501742, 1.6635005917178431, 0.24113763858859116, -1.2898772197929023], [-2.23985985733975, 1.2765709163818362, 0.5680313489899385, 0.847296954372218], [-0.3328748962507801, 0.34482939616746583, 1.8823816030932006, 0.33483420185153473]], [[-1.1155186762678087, -0.43144971563468354, 1.385110615586996, -1.089961064286938], [-2.58142946001913, -2.293955187945487, -1.3389389586186435, 0.7993935284932526], [0.08846320092734136, -0.7572771271592771, -1.2851365294184347, -0.11398930813306221]]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[[0.43890868660529964, 0.28057911287523135, 0.2548077614781974, 0.5555517930622165], [0.3935955784321085, 0.5316050463499963, 0.3360450945466618, 0.1536317826938386], [0.16749573496259176, 0.18781584077477248, 0.4091471439751409, 0.2908164242439449]], [[0.10392484703017076, 0.22281962036655667, 0.9173319897200162, 0.06097146139982382], [0.686145278575895, 0.5394865175086676, 0.07439874115203143, 0.033959679098575994], [0.20992987439393418, 0.23769386212477572, 0.008269269127952365, 0.9050688595016003]]], [[[0.3840025952757096, 0.12021598018944052, 0.2566473575157578, 0.15499605871958333], [0.39363920587152357, 0.0887787631517361, 0.5079203735779203, 0.08524665711003312], [0.2223581988527668, 0.7910052566588234, 0.23543226890632185, 0.7597572841703835]], [[0.3251701822340204, 0.512462019203141, 0.6667019760631633, 0.42591798347740784], [0.17936621609156714, 0.2260576873623843, 0.23905356632177419, 0.21587447919355537], [0.49546360167441245, 0.2614802934344747, 0.09424445761506242, 0.3582075373290369]]]], "dtype": "float64"}, {"__nd__": [[[[0.7789367657857675, 0.482871255420624, 0.4081026226934998, 0.5692063785887429], [-0.6304732567107356, -0.8360594266628745, -0.13546345807495594, -0.46181943819804144], [-0.14846350907503192, 0.3531881712422504, -0.27263916461854376, -0.1073869403907015]], [[-0.13624366640574992, -0.025203366996359885, 0.09315149592093819, -0.03559990382291206], [-0.051939195057700296, -0.10711526935347902, -0.1117613158767351, -0.025061940559941306], [0.18818286146345023, 0.13231863634983893, 0.018609819955796945, 0.06066184438285329]]], [[[0.5388291714734654, 0.12952394114298887, -0.1417816928100031, -0.21956327703561945], [-0.6152957616314522, 0.06130149715489086, -0.11455842092687644, 0.061428816376609845], [0.0764665901579868, -0.1908254382978797, 0.25634011373687954, 0.1581344606590096]], [[-0.10847492804316677, 0.2594242254976198, 0.6019317187003823, -0.32261748747599395], [-0.32277042251999283, -0.30659624194236784, -0.43536431070658166, 0.2443463185565578], [0.43124535056315966, 0.04717201644474808, -0.16656740799380051, 0.07827116891943617]]]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(softmax_and_grad, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")